In [ ]:
# all_data va contenir TOUTES les reponses JSON chargees depuis data/raw
all_data = []

# file_path prend chaque chemin de fichier JSON un par un
for file_path in json_files:
    # Ouvre le fichier en lecture texte UTF-8
    with file_path.open("r", encoding="utf-8") as f:
        # json.load transforme le contenu JSON en dictionnaire Python
        payload = json.load(f)
    # On ajoute le dictionnaire de la ville dans la liste globale
    all_data.append(payload)

print(f"Fichiers charges: {len(all_data)}")
print("Villes:", [item.get("name") for item in all_data])

# Affiche toutes les donnees brutes (liste de dictionnaires)
all_data

Fichiers charges: 5
Villes: ['Berlin', 'Londres', 'Madrid', 'Paris', 'Rome']


[{'coord': {'lon': 13.4105, 'lat': 52.5244},
  'weather': [{'id': 800,
    'main': 'Clear',
    'description': 'ciel dégagé',
    'icon': '01d'}],
  'base': 'stations',
  'main': {'temp': 9.53,
   'feels_like': 9.07,
   'temp_min': 7.31,
   'temp_max': 10.6,
   'pressure': 1017,
   'humidity': 44,
   'sea_level': 1017,
   'grnd_level': 1012},
  'visibility': 10000,
  'wind': {'speed': 1.54, 'deg': 180},
  'clouds': {'all': 0},
  'dt': 1775824686,
  'sys': {'type': 2,
   'id': 2011538,
   'country': 'DE',
   'sunrise': 1775794809,
   'sunset': 1775843692},
  'timezone': 7200,
  'id': 2950159,
  'name': 'Berlin',
  'cod': 200},
 {'coord': {'lon': -0.1257, 'lat': 51.5085},
  'weather': [{'id': 804,
    'main': 'Clouds',
    'description': 'couvert',
    'icon': '04d'}],
  'base': 'stations',
  'main': {'temp': 12.69,
   'feels_like': 11.19,
   'temp_min': 11.86,
   'temp_max': 13.88,
   'pressure': 1019,
   'humidity': 45,
   'sea_level': 1019,
   'grnd_level': 1015},
  'visibility': 1000

In [ ]:
# resultats contiendra une version "propre" des donnees: 1 dictionnaire par ville
resultats = []

# data represente une reponse meteo complete pour UNE ville
for data in all_data:
    # On extrait uniquement les champs utiles du JSON brut
    resultat = {
        "ville": data.get("name"),
        "temperature": data.get("main", {}).get("temp"),
        "ressenti": data.get("main", {}).get("feels_like"),
        "humidite": data.get("main", {}).get("humidity"),
        "vitesse_vent": data.get("wind", {}).get("speed"),
        "description_meteo": (data.get("weather") or [{}])[0].get("description")
    }
    # On ajoute ce resume a la liste finale
    resultats.append(resultat)

# Affiche la liste de resumes
resultats

[{'ville': 'Berlin',
  'temperature': 9.53,
  'ressenti': 9.07,
  'humidite': 44,
  'vitesse_vent': 1.54,
  'description_meteo': 'ciel dégagé'},
 {'ville': 'Londres',
  'temperature': 12.69,
  'ressenti': 11.19,
  'humidite': 45,
  'vitesse_vent': 3.6,
  'description_meteo': 'couvert'},
 {'ville': 'Madrid',
  'temperature': 25.91,
  'ressenti': 25.21,
  'humidite': 25,
  'vitesse_vent': 3.6,
  'description_meteo': 'ciel dégagé'},
 {'ville': 'Paris',
  'temperature': 15.52,
  'ressenti': 14.2,
  'humidite': 41,
  'vitesse_vent': 2.06,
  'description_meteo': 'ciel dégagé'},
 {'ville': 'Rome',
  'temperature': 4.98,
  'ressenti': 3.87,
  'humidite': 87,
  'vitesse_vent': 1.54,
  'description_meteo': 'ciel dégagé'}]

In [13]:
import pandas as pd

In [14]:
df=pd.DataFrame(resultats)

In [15]:
df

,ville,temperature,ressenti,humidite,vitesse_vent,description_meteo
0,Berlin,9.53,9.07,44,1.54,ciel dégagé
1,Londres,12.69,11.19,45,3.60,couvert
2,Madrid,25.91,25.21,25,3.60,ciel dégagé
3,Paris,15.52,14.20,41,2.06,ciel dégagé
4,Rome,4.98,3.87,87,1.54,ciel dégagé


In [16]:
scores=[]

In [ ]:
# Liste qui stocke les scores de confort (meme ordre que les lignes du DataFrame)
scores = []

# df.iterrows() parcourt le DataFrame ligne par ligne
# A chaque tour:
# - '_' est l'index de ligne (0, 1, 2, ...). On ne l'utilise pas, donc on met '_'.
# - 'row' est une Series pandas qui contient les valeurs de la ligne courante.
for _, row in df.iterrows():
    # Score de base sur 10
    score = 10

    # row["temperature"] lit la colonne 'temperature' de la ligne courante
    # row["humidite"] lit la colonne 'humidite' de la ligne courante
    temperature = row["temperature"]
    humidite = row["humidite"]

    # Penalite temperature: trop froid ou trop chaud => plus grosse baisse
    if temperature < 10 or temperature > 32:
        score -= 4
    elif 10 <= temperature < 18 or 24 < temperature <= 32:
        score -= 2

    # Penalite humidite: tres sec ou tres humide => inconfort
    if humidite < 30 or humidite > 80:
        score -= 3
    elif 30 <= humidite < 40 or 60 < humidite <= 80:
        score -= 1.5

    # On borne le score pour rester entre 0 et 10
    score = max(0, min(10, score))

    # On stocke le score de cette ligne
    scores.append(score)

# Ajoute la nouvelle colonne au DataFrame
df["indice_confort"] = scores

# Affiche le tableau final
df

,ville,temperature,ressenti,humidite,vitesse_vent,description_meteo,indice_confort
0,Berlin,9.53,9.07,44,1.54,ciel dégagé,6
1,Londres,12.69,11.19,45,3.60,couvert,8
2,Madrid,25.91,25.21,25,3.60,ciel dégagé,5
3,Paris,15.52,14.20,41,2.06,ciel dégagé,8
4,Rome,4.98,3.87,87,1.54,ciel dégagé,3


In [ ]:
# Sauvegarde le DataFrame en CSV dans le dossier courant du notebook
# index=False evite d'ajouter une colonne d'index inutile dans le fichier
df.to_csv("df.csv", index=False)

# Message de confirmation
print("CSV cree: df.csv")